# AquaMark AI Theory Notebook

This notebook documents the current AquaMark runtime built around a TensorFlow encoder/decoder watermarking stack.

For setup, commands, and endpoint usage, see [README.md](./README.md). This notebook is the conceptual companion to that operational guide.


## 1. System Snapshot

The active project pipeline is:

1. A user uploads an image in the React frontend.
2. The FastAPI backend converts watermark text into a fixed-length bit vector.
3. The encoder produces a visually stable watermarked image.
4. The decoder estimates watermark bits and integrity probabilities from a suspect image.
5. The UI surfaces model status, including whether trained checkpoints are available.

The runtime entrypoint is `backend/main.py`, not the older SVM reference code still present in the repository.


## 2. Mathematical Framing

Let:

- `I` be the input image tensor.
- `b` be the watermark bit vector derived from text.
- `s` be the embedding strength scalar.
- `E(I, b, s)` be the encoder.
- `D(I')` be the decoder applied to a clean or attacked image.

The embedding stage can be summarized as:

`I_w = E(I, b, s)`

The verification stage predicts both recovered bits and integrity probabilities:

`D(I') -> (p_bits, p_integrity)`

Recovered bits are thresholded from `p_bits`, and bit error rate is computed against the expected watermark bit vector.


## 3. Training Pipeline

The current training loop is implemented in `backend/models/training_pipeline.py`.

Its responsibilities include:

- sampling cover images from the synthetic cover repository,
- generating random watermark bit vectors,
- producing clean and attacked variants of watermarked images,
- optimizing reconstruction, perceptual, SSIM, consistency, and integrity losses,
- exporting `encoder.weights.h5`, `decoder.weights.h5`, and checkpoint metadata.

Attack simulation is part of training, which is why the system is described as attack-aware rather than simple watermark insertion plus post-hoc classification.


## 4. Runtime Status and Reliability

AquaMark now exposes `/api/model-status` so the frontend can distinguish between two states:

- trained checkpoints are loaded, or
- the app is running with baseline-initialized models.

This is important because the application can still boot and process requests when checkpoints are missing, but verification quality is not yet trustworthy in that state.

Operational instructions for checking and improving this state are documented in [README.md](./README.md).


## 5. File-to-Responsibility Map

- `backend/main.py`: application lifecycle, CORS, static outputs, and router registration.
- `backend/train.py`: CLI entrypoint for checkpoint generation.
- `backend/services/embed_service.py`: image ingestion, embedding, previews, and output export.
- `backend/services/verify_service.py`: decoder inference, BER analysis, and integrity state selection.
- `frontend/src/App.jsx`: fetches shared model status for the UI.
- `frontend/src/pages/EmbedPage.jsx` and `frontend/src/pages/VerifyPage.jsx`: main user workflows.

The README is the fastest place to look for commands and route details; this notebook is the best place to keep the conceptual map of why those pieces exist.


## 6. Documentation Contract

These two docs should move together:

- Update `README.md` when commands, routes, startup behavior, or project layout change.
- Update `theory.ipynb` when the active model architecture, training logic, or verification interpretation changes.

Return to [README.md](./README.md) for the practical runbook.
